# Retail Sales Analytics — End-to-End Capstone

**Important data note:** The included dataset is synthetic and generated for this assignment demonstration. It is not real company data. Replace it with an approved real-world dataset if your internship evaluator requires real data, then rerun all cells and update conclusions.

This notebook performs inspection, cleaning, EDA, chart generation, and business analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path('../data')
OUT = Path('../visualizations')
OUT.mkdir(exist_ok=True)
raw = pd.read_csv(DATA / 'raw_sales.csv')
print('Raw shape:', raw.shape)
display(raw.head())
display(raw.dtypes)
display(raw.isna().sum().sort_values(ascending=False))
print('Duplicate rows:', raw.duplicated().sum())

## Data cleaning

Remove duplicate records and invalid quantities/discounts. Fill categorical nulls with mode and missing unit prices with the median. Recompute sales from unit price × quantity × (1 − discount).

In [ ]:
df = raw.copy()
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
for c in ['Quantity','Unit_Price','Discount','Sales','Profit']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.drop_duplicates()
df = df.dropna(subset=['Order_ID','Order_Date','Category','Product'])
df = df[(df['Quantity'] > 0) & (df['Unit_Price'] >= 0) & df['Discount'].between(0,1)]
for c in ['Region','Customer_Segment','Payment_Method']:
    df[c] = df[c].fillna(df[c].mode().iloc[0])
df['Unit_Price'] = df['Unit_Price'].fillna(df['Unit_Price'].median())
df['Discount'] = df['Discount'].fillna(df['Discount'].median())
df['Sales'] = (df['Unit_Price'] * df['Quantity'] * (1-df['Discount'])).round(2)
margin_by_cat = (df['Profit'] / df['Sales'].replace(0,np.nan)).groupby(df['Category']).median()
global_margin = (df['Profit'] / df['Sales'].replace(0,np.nan)).median()
mask = df['Profit'].isna()
df.loc[mask,'Profit'] = df.loc[mask].apply(lambda r: r['Sales']*margin_by_cat.get(r['Category'],global_margin), axis=1)
df['Profit'] = df['Profit'].round(2)
df.to_csv(DATA / 'cleaned_sales_from_notebook.csv', index=False)
print('Cleaned shape:', df.shape)
print('Remaining missing values:', int(df.isna().sum().sum()))
display(df.head())

## Descriptive statistics and key metrics

In [ ]:
display(df.describe(include='all').T)
print('Total sales:', round(df['Sales'].sum(),2))
print('Total profit:', round(df['Profit'].sum(),2))
print('Unique orders:', df['Order_ID'].nunique())
print('Total units:', int(df['Quantity'].sum()))
print('Profit margin %:', round(100*df['Profit'].sum()/df['Sales'].sum(),2))
print('Sales by category:')
display(df.groupby('Category').agg(Orders=('Order_ID','nunique'), Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales',ascending=False))

## Charts (7 visualizations)

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Month'] = df['Order_Date'].dt.to_period('M').astype(str)
fig, ax = plt.subplots(figsize=(11,4)); df.groupby('Month')['Sales'].sum().plot(ax=ax, marker='o'); ax.set(title='Monthly Sales Trend', xlabel='Month', ylabel='Sales'); plt.xticks(rotation=60); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8,4)); df.groupby('Category')['Sales'].sum().sort_values().plot(kind='barh', ax=ax); ax.set(title='Sales by Category', xlabel='Sales', ylabel='Category'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8,4)); df.groupby('Region')['Sales'].sum().sort_values().plot(kind='bar', ax=ax); ax.set(title='Sales by Region', xlabel='Region', ylabel='Sales'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8,4)); df.groupby('Product')['Sales'].sum().nlargest(10).sort_values().plot(kind='barh', ax=ax); ax.set(title='Top 10 Products by Sales', xlabel='Sales', ylabel='Product'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); ax.scatter(df['Sales'],df['Profit'],alpha=.5); ax.set(title='Sales vs Profit', xlabel='Sales', ylabel='Profit'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); sns.heatmap(df[['Quantity','Unit_Price','Discount','Sales','Profit']].corr(), annot=True, cmap='Blues', ax=ax); ax.set_title('Numeric Correlation Heatmap'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4)); df.groupby('Customer_Segment')['Sales'].sum().sort_values().plot(kind='bar',ax=ax); ax.set(title='Sales by Customer Segment',xlabel='Segment',ylabel='Sales'); plt.tight_layout(); plt.show()

## Outlier screening and interpretation

Use the IQR rule for numeric columns as a screening method; investigate flagged values rather than deleting automatically. Sales and profit are expected to be right-skewed in many retail datasets. Correlation does not prove causation. Discounts may be associated with sales/profit patterns, but the relationship must be interpreted in context.

In [ ]:
for col in ['Quantity','Unit_Price','Discount','Sales','Profit']:
    q1, q3 = df[col].quantile([.25,.75]); iqr=q3-q1
    low, high = q1-1.5*iqr, q3+1.5*iqr
    print(f'{col}: {((df[col]<low)|(df[col]>high)).sum()} IQR-flagged rows')
print('Correlation matrix:')
display(df[['Quantity','Unit_Price','Discount','Sales','Profit']].corr().round(3))